In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import sys
sys.path.append('../')

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch

In [ ]:
import pickle
import random
import h5py

from torch.utils.data import IterableDataset
from torch_geometric.data import Data, DataLoader

from Utils import AU2EV, rmsd_loss, d_mae_loss, Kabsch_alignment, pairwise_dist_to_coord, generate_fully_connected, count_negative_eig, xyz2AC

In [ ]:
import torch_geometric

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Geometry import Point3D

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
def get_molecular_reference_energy(atomic_numbers):
    molecular_reference_energy = 0
    for atomic_number in atomic_numbers:
        molecular_reference_energy += REFERENCE_ENERGIES[atomic_number]

    return molecular_reference_energy

def generator(formula, rxn, grp):
    """ Iterates through a h5 group """

    energies = grp["wB97x_6-31G(d).energy"]
    forces = grp["wB97x_6-31G(d).forces"]
    atomic_numbers = list(grp["atomic_numbers"])
    positions = grp["positions"]
    molecular_reference_energy = get_molecular_reference_energy(atomic_numbers)

    for energy, force, positions in zip(energies, forces, positions):
        d = {
            "rxn": rxn,
            "wB97x_6-31G(d).energy": energy.__float__(),
            "wB97x_6-31G(d).atomization_energy": energy
            - molecular_reference_energy.__float__(),
            "wB97x_6-31G(d).forces": force.tolist(),
            "positions": positions,
            "formula": formula,
            "atomic_numbers": atomic_numbers,
        }

        yield d

def get_dynamics_data(formula, rxn, data):
    reactant = next(generator(formula, rxn, data[formula][rxn]["reactant"]))
    product = next(generator(formula, rxn, data[formula][rxn]["product"]))
    transition_state = next(generator(formula, rxn, data[formula][rxn]["transition_state"]))
    x = torch.tensor(reactant['atomic_numbers'], dtype=torch.long)

    reactant_pos = torch.tensor(reactant['positions'], dtype=torch.float32)
    product_pos = torch.tensor(product['positions'], dtype=torch.float32)
    transition_state_pos = torch.tensor(transition_state['positions'], dtype=torch.float32)

    energies = list()
    energies.append(torch.tensor(reactant['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(product['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(transition_state['wB97x_6-31G(d).energy'], dtype=torch.float32))
    
    return Data(x=x, reactant_pos=reactant_pos, product_pos=product_pos, transition_state_pos=transition_state_pos, energies=torch.stack(energies))

class Dataset_dynamics(IterableDataset):
    def __init__(self, hdf5_file, datasplit):
        super(Dataset_dynamics, self).__init__()
        self.hdf5_file = hdf5_file
        self.datasplit = datasplit
        assert datasplit in [
            "train",
            "valid",
            "test",
        ]
        with open('../Data/reactions_'+self.datasplit+'.pickle', 'rb') as f:
            self.datalist = pickle.load(f)

    def __iter__(self):
        with h5py.File(self.hdf5_file, "r") as f:
            data = f['data']
            i = 0
            if self.datasplit == 'train':
                random.shuffle(self.datalist)
            for formula, rxn in self.datalist:
                yield get_dynamics_data(formula, rxn, data)
                    
    def __len__(self):
        pass
    
def generate_dataloader_dynamics(hdf5_file, batch_size):
    dataloaders = {}
    dataloaders['train'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'train'), batch_size = batch_size)
    dataloaders['val'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'valid'), batch_size = batch_size)
    dataloaders['test'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'test'), batch_size = batch_size)
    return dataloaders

In [ ]:
dataloader = generate_dataloader_dynamics('transition1x.h5', 1)

In [ ]:
def get_positions_from_pyscf(pyscf_dft_res):
    atom_type_and_pos = pyscf_dft_res[0].mol.atom
    atom_pos = [temp[1] for temp in atom_type_and_pos]
    return np.vstack(atom_pos)

In [ ]:
def bond_analysis(x, true_pos_list, pred_pos_list, reactant_list, product_list):
    true_bond_length = []
    pred_bond_length = []
    bond_type = []
    batch = []
    for i in range(len(true_pos_list)):
        src, dst = generate_fully_connected(torch.zeros(reactant_list[i].shape[0], dtype=torch.long))
        batch_temp = torch.zeros_like(src) + i
        batch.append(batch_temp)
        reactant = torch.tensor(reactant_list[i], device='cpu')
        product = torch.tensor(product_list[i], device='cpu')
        true_trans = torch.tensor(true_pos_list[i], device='cpu')
        pred_trans = torch.tensor(pred_pos_list[i], device='cpu')
        reactant_bond_dist = torch.norm(reactant[src] - reactant[dst], dim=-1, p=2)
        product_bond_dist = torch.norm(product[src] - product[dst], dim=-1, p=2)
        truets_bond_dist = torch.norm(true_trans[src] - true_trans[dst], dim=-1, p=2)
        predts_bond_dist = torch.norm(pred_trans[src] - pred_trans[dst], dim=-1, p=2)
        bond_type_list = torch.zeros_like(src)
        bond_reactant, _ = xyz2AC(x[i].numpy().tolist(), reactant.numpy().tolist(), 0)
        bond_product, _ = xyz2AC(x[i].numpy().tolist(), product.numpy().tolist(), 0)
        bond_reactant = bond_reactant[src, dst]
        bond_product = bond_product[src, dst]
        bond_type_list[(bond_reactant == 1) & (bond_product == 1)] = 1
        bond_type_list[(bond_reactant == 0) & (bond_product == 1)] = 2
        bond_type_list[(bond_reactant == 1) & (bond_product == 0)] = 3
        bond_type_list[(bond_reactant == 0) & (bond_product == 0)] = 4
        true_bond_length.append(truets_bond_dist)
        pred_bond_length.append(predts_bond_dist)
        bond_type.append(bond_type_list)

    return torch.cat(true_bond_length), torch.cat(pred_bond_length), torch.cat(bond_type), torch.cat(batch)

Baseline NeuralNEB

In [ ]:
with open('res_neuralneb.pickle', 'rb') as f:
    temp = pickle.load(f)
dft_res_neb = temp['neb']['dft_res']
pred_trans_pos_neb = [get_positions_from_pyscf(temp_res) for temp_res in dft_res_neb]
true_trans_pos = [data.transition_state_pos for data in dataloader['test']]
reactant_pos = [data.reactant_pos for data in dataloader['test']]
product_pos = [data.product_pos for data in dataloader['test']]
x = [data.x for data in dataloader['test']]
res_neuralNEB = bond_analysis(x, true_trans_pos, pred_trans_pos_neb, reactant_pos, product_pos)

Baselines learnTS (PSI-based)

In [ ]:
with open('res_learnts.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['pred_transition_state_pos']
true_trans_pos = temp['true_transition_state_pos']
reactants, products = [reactant for reactant, product in temp['reactant_product_pos']], [product for reactant, product in temp['reactant_product_pos']]
x = temp['atom_types']

In [ ]:
res_learnTS = bond_analysis(x, true_trans_pos, pred_trans_pos_model, reactants, products)

Baseline OA-Reactdiff

In [ ]:
with open('res_reactdiff.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['pred_transition_state_pos']
true_trans_pos = temp['true_transition_state_pos']
reactants, products = [reactant for reactant, product in temp['reactant_product_pos']], [product for reactant, product in temp['reactant_product_pos']]
x = [types.cpu() for types in temp['atom_types']]

In [ ]:
res_reactdiff = bond_analysis(x, true_trans_pos[:-1], pred_trans_pos_model, reactants, products)

Baseline react-ot

In [ ]:
with open('res_reactot.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['pred_transition_state_pos']
true_trans_pos = temp['true_transition_state_pos']
reactants, products = [reactant for reactant, product in temp['reactant_product_pos']], [product for reactant, product in temp['reactant_product_pos']]
x = [torch.tensor(types) for types in temp['atom_types']]

In [ ]:
res_reactot = bond_analysis(x, true_trans_pos, pred_trans_pos_model, reactants, products)

Our Method

In [ ]:
with open('res_fm.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['fm']['pred_trans_pos']
true_trans_pos = [data.transition_state_pos for data in dataloader['test']]
reactant_pos = [data.reactant_pos for data in dataloader['test']]
product_pos = [data.product_pos for data in dataloader['test']]
x = [data.x for data in dataloader['test']]

In [ ]:
res_ours = bond_analysis(x, true_trans_pos, pred_trans_pos_model, reactants, products)

In [ ]:
matplotlib.rcParams.update({'font.size': 8})
matplotlib.rcParams.update({'axes.titlesize': 6})
matplotlib.rcParams.update({'axes.labelsize': 6})
matplotlib.rcParams.update({'legend.fontsize': 6})
matplotlib.rcParams.update({'xtick.labelsize': 6})
matplotlib.rcParams.update({'ytick.labelsize': 6})
matplotlib.rcParams.update({'axes.linewidth': 1.0, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8, 'xtick.minor.width': 0.6, 'ytick.minor.width': 0.6, 'grid.linewidth': 0.5})
matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Arial'})
# matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Times New Roman'})

In [ ]:
width = 180 / 25.4
height = 90 / 25.4
figs = plt.figure(figsize=(width, height))
subfigs = figs.subfigures(1, 2, width_ratios=[1, 1], wspace=0.2)

In [ ]:
fig = subfigs[0]
fig.text(-0.12, 0.95, 'd', ha='center', va='center', fontsize=10)
ax_box = fig.subplots(nrows=4, sharex=True, gridspec_kw={'height_ratios': [1, 1, 1, 1], 'hspace': 0.0})

In [ ]:
ax = ax_box[0]

data = pd.DataFrame([
    pd.Series((torch.abs(res_neuralNEB[0] - res_neuralNEB[1])/res_neuralNEB[0])[res_neuralNEB[2]==1], name = 'NeuralNEB'),
    pd.Series((torch.abs(res_learnTS[0] - res_learnTS[1])/res_learnTS[0])[res_learnTS[2]==1], name = 'PSI-based'),
    pd.Series((torch.abs(res_reactdiff[0] - res_reactdiff[1])/res_reactdiff[0])[res_reactdiff[2]==1], name = 'OA-ReactDiff'),
    pd.Series((torch.abs(res_reactot[0] - res_reactot[1])/res_reactot[0])[res_reactot[2]==1], name = 'React-OT'),
    pd.Series((torch.abs(res_ours[0] - res_ours[1])/res_ours[0])[res_ours[2]==1], name = 'TS-DFM')
]).transpose()

data_length = pd.DataFrame([
    pd.Series(res_neuralNEB[0][res_neuralNEB[2]==1], name = 'NeuralNEB'),
    pd.Series(res_learnTS[0][res_learnTS[2]==1], name = 'PSI-based'),
    pd.Series(res_reactdiff[0][res_reactdiff[2]==1], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[0][res_reactot[2]==1], name = 'React-OT'),
    pd.Series(res_ours[0][res_ours[2]==1], name = 'TS-DFM')
]).transpose()


sns.boxplot(data=data, ax=ax, palette='Set2', orient='h', whis=100.0, boxprops=dict(alpha=0.7), width=0.6)

ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)
ax.set_xscale('log')
ax.set_xlim([1e-3, 1e0])
ax.set_xticks([1e-3, 2e-3, 5e-3, 1e-2, 2e-2, 5e-2, 1e-1, 2e-1, 5e-1, 1e0])
ax.set_xticklabels(['0.1%', '0.2%', '0.5%', '1%', '2%', '5%', '10%', '20%', '50%', '100%'])
ax.set_ylabel('Unchanged\n(bonded)\n['+str((torch.sum(res_neuralNEB[2]==1)/res_neuralNEB[2].shape[0]).item()*100.0)[:5]+'%]', rotation=0, va='center', labelpad=15)

legend_handles = [matplotlib.patches.Patch(color=color, label=label) for color, label in zip(sns.color_palette('Set2')[:5], ['NeuralNEB', 'PSI-based', 'OA-ReactDiff', 'React-OT', 'TS-DFM'])]
ax.legend(handles=legend_handles, loc='upper right')

ax.set_yticklabels([])

In [ ]:
ax = ax_box[1]

data = pd.DataFrame([
    pd.Series((torch.abs(res_neuralNEB[0] - res_neuralNEB[1])/res_neuralNEB[0])[res_neuralNEB[2]==2], name = 'NeuralNEB'),
    pd.Series((torch.abs(res_learnTS[0] - res_learnTS[1])/res_learnTS[0])[res_learnTS[2]==2], name = 'PSI-based'),
    pd.Series((torch.abs(res_reactdiff[0] - res_reactdiff[1])/res_reactdiff[0])[res_reactdiff[2]==2], name = 'OA-ReactDiff'),
    pd.Series((torch.abs(res_reactot[0] - res_reactot[1])/res_reactot[0])[res_reactot[2]==2], name = 'React-OT'),
    pd.Series((torch.abs(res_ours[0] - res_ours[1])/res_ours[0])[res_ours[2]==2], name = 'TS-DFM')
]).transpose()

data_length = pd.DataFrame([
    pd.Series(res_neuralNEB[0][res_neuralNEB[2]==2], name = 'NeuralNEB'),
    pd.Series(res_learnTS[0][res_learnTS[2]==2], name = 'PSI-based'),
    pd.Series(res_reactdiff[0][res_reactdiff[2]==2], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[0][res_reactot[2]==2], name = 'React-OT'),
    pd.Series(res_ours[0][res_ours[2]==2], name = 'TS-DFM')
]).transpose()

sns.boxplot(data=data, ax=ax, palette='Set2', orient='h', whis=20.0, boxprops=dict(alpha=0.7), width=0.6)

ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)
ax.set_xscale('log')
ax.set_xlim([1e-3, 1e0])
ax.set_xticks([1e-3, 2e-3, 5e-3, 1e-2, 2e-2, 5e-2, 1e-1, 2e-1, 5e-1, 1e0])
ax.set_xticklabels(['0.1%', '0.2%', '0.5%', '1%', '2%', '5%', '10%', '20%', '50%', '100%'])
ax.set_ylabel('Bond\nFormation\n['+str((torch.sum(res_neuralNEB[2]==2)/res_neuralNEB[2].shape[0]).item()*100.0)[:5]+'%]', rotation=0, va='center', labelpad=15)

ax.set_yticklabels([])

In [ ]:
ax = ax_box[2]

data = pd.DataFrame([
    pd.Series((torch.abs(res_neuralNEB[0] - res_neuralNEB[1])/res_neuralNEB[0])[res_neuralNEB[2]==3], name = 'NeuralNEB'),
    pd.Series((torch.abs(res_learnTS[0] - res_learnTS[1])/res_learnTS[0])[res_learnTS[2]==3], name = 'PSI-based'),
    pd.Series((torch.abs(res_reactdiff[0] - res_reactdiff[1])/res_reactdiff[0])[res_reactdiff[2]==3], name = 'OA-ReactDiff'),
    pd.Series((torch.abs(res_reactot[0] - res_reactot[1])/res_reactot[0])[res_reactot[2]==3], name = 'React-OT'),
    pd.Series((torch.abs(res_ours[0] - res_ours[1])/res_ours[0])[res_ours[2]==3], name = 'TS-DFM')
]).transpose()

data_length = pd.DataFrame([
    pd.Series(res_neuralNEB[0][res_neuralNEB[2]==3], name = 'NeuralNEB'),
    pd.Series(res_learnTS[0][res_learnTS[2]==3], name = 'PSI-based'),
    pd.Series(res_reactdiff[0][res_reactdiff[2]==3], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[0][res_reactot[2]==3], name = 'React-OT'),
    pd.Series(res_ours[0][res_ours[2]==3], name = 'TS-DFM')
]).transpose()

sns.boxplot(data=data, ax=ax, palette='Set2', orient='h', whis=50.0, boxprops=dict(alpha=0.7), width=0.6)

ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)
ax.set_xscale('log')
ax.set_xlim([1e-3, 1e0])
ax.set_xticks([1e-3, 2e-3, 5e-3, 1e-2, 2e-2, 5e-2, 1e-1, 2e-1, 5e-1, 1e0])
ax.set_xticklabels(['0.1%', '0.2%', '0.5%', '1%', '2%', '5%', '10%', '20%', '50%', '100%'])
ax.set_ylabel('Bond\nBreaking\n['+str((torch.sum(res_neuralNEB[2]==3)/res_neuralNEB[2].shape[0]).item()*100.0)[:5]+'%]', rotation=0, va='center', labelpad=15)

ax.set_yticklabels([])

In [ ]:
ax = ax_box[3]

data = pd.DataFrame([
    pd.Series((torch.abs(res_neuralNEB[0] - res_neuralNEB[1])/res_neuralNEB[0])[res_neuralNEB[2]==4], name = 'NeuralNEB'),
    pd.Series((torch.abs(res_learnTS[0] - res_learnTS[1])/res_learnTS[0])[res_learnTS[2]==4], name = 'PSI-based'),
    pd.Series((torch.abs(res_reactdiff[0] - res_reactdiff[1])/res_reactdiff[0])[res_reactdiff[2]==4], name = 'OA-ReactDiff'),
    pd.Series((torch.abs(res_reactot[0] - res_reactot[1])/res_reactot[0])[res_reactot[2]==4], name = 'React-OT'),
    pd.Series((torch.abs(res_ours[0] - res_ours[1])/res_ours[0])[res_ours[2]==4], name = 'TS-DFM')
]).transpose()

data_length = pd.DataFrame([
    pd.Series(res_neuralNEB[0][res_neuralNEB[2]==4], name = 'NeuralNEB'),
    pd.Series(res_learnTS[0][res_learnTS[2]==4], name = 'PSI-based'),
    pd.Series(res_reactdiff[0][res_reactdiff[2]==4], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[0][res_reactot[2]==4], name = 'React-OT'),
    pd.Series(res_ours[0][res_ours[2]==4], name = 'TS-DFM')
]).transpose()


sns.boxplot(data=data, ax=ax, palette='Set2', orient='h', whis=50.0, boxprops=dict(alpha=0.7), width=0.6)

ax.tick_params(axis='both', which='both', bottom=True, top=False, left=False, right=False)
ax.set_xscale('log')
ax.set_xlim([1e-3, 1e0])
ax.set_xticks([1e-3, 2e-3, 5e-3, 1e-2, 2e-2, 5e-2, 1e-1, 2e-1, 5e-1, 1e0])
ax.set_xticklabels(['0.1%', '0.2%', '0.5%', '1%', '2%', '5%', '10%', '20%', '50%', '100%'])
ax.set_ylabel('Unchanged\n(non-bonded)\n['+str((torch.sum(res_neuralNEB[2]==4)/res_neuralNEB[2].shape[0]).item()*100.0)[:5]+'%]', rotation=0, va='center', labelpad=15)
ax.set_xlabel('Distance Absolute Percentage Error')

ax.set_yticklabels([])

In [ ]:
fig = subfigs[1]
fig.text(-0.12, 0.95, 'e', ha='center', va='center', fontsize=10)
ax_dist = fig.add_subplot()

In [ ]:
data = pd.DataFrame([
    pd.Series((torch.abs(res_neuralNEB[0] - res_neuralNEB[1])/res_neuralNEB[0]), name = 'NeuralNEB'),
    pd.Series((torch.abs(res_learnTS[0] - res_learnTS[1])/res_learnTS[0]), name = 'PSI-based'),
    pd.Series((torch.abs(res_reactdiff[0] - res_reactdiff[1])/res_reactdiff[0]), name = 'OA-ReactDiff'),
    pd.Series((torch.abs(res_reactot[0] - res_reactot[1])/res_reactot[0]), name = 'React-OT'),
    pd.Series((torch.abs(res_ours[0] - res_ours[1])/res_ours[0]), name = 'TS-DFM')
]).transpose()

data_length = pd.DataFrame([
    pd.Series(res_neuralNEB[0], name = 'NeuralNEB'),
    pd.Series(res_learnTS[0], name = 'PSI-based'),
    pd.Series(res_reactdiff[0], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[0], name = 'React-OT'),
    pd.Series(res_ours[0], name = 'TS-DFM')
]).transpose()

sns.histplot(data=data_length.iloc[:, 0], ax=ax_dist, element="step",
             color='lightgray', alpha=0.7, stat='count', bins=100, kde=False, label='Interatomic Distance\nDistribution')

ax_dist.set_xlabel(r'Interatomic Distance (Å)')

ax2 = ax_dist.twinx()


bins = np.linspace(data_length.iloc[:, 0].min(), data_length.iloc[:, 0].max(), 100)


for i, method in enumerate(data):

    df = pd.DataFrame({
        'bond_length': data_length.iloc[:, i],
        method: data.iloc[:, i]
    })
    df['bin'] = pd.cut(df['bond_length'], bins=bins)
    bin_means = df.groupby('bin')[method].mean()
    
    x_positions = [(bin.left + bin.right)/2 for bin in bin_means.index]
    ax2.plot(x_positions, bin_means.values, color=sns.color_palette('Set2')[i], 
            linewidth=1.5, label=f'{method}')
    
ax2.legend(loc='upper center')
ax_dist.legend(loc='upper right')
ax2.set_xlim([0.0, 6.0])
ax2.set_yscale('log')
ax2.set_ylim([1e-3, 1e0])
ax2.set_yticks([1e-3, 1e-2, 1e-1, 1e0])
ax2.set_ylabel('Mean Distance Absolute Percentage Error')
ax2.set_yticklabels(['0.1%', '1%', '10%', '100%'])

In [ ]:
figs.tight_layout()
figs.savefig('res_bond_analysis.eps', bbox_inches='tight', dpi=1200)
plt.show()

In [ ]:
import pandas as pd

res_neuralNEB = pd.DataFrame({
    'true_bond_length': res_neuralNEB[0].numpy(),
    'pred_bond_length': res_neuralNEB[1].numpy(),
    'bond_type': res_neuralNEB[2].numpy()})
res_learnTS = pd.DataFrame({
    'true_bond_length': res_learnTS[0].numpy(),
    'pred_bond_length': res_learnTS[1].numpy(),
    'bond_type': res_learnTS[2].numpy()})
res_reactdiff = pd.DataFrame({
    'true_bond_length': res_reactdiff[0].numpy(),
    'pred_bond_length': res_reactdiff[1].numpy(),
    'bond_type': res_reactdiff[2].numpy()})
res_reactot = pd.DataFrame({
    'true_bond_length': res_reactot[0].numpy(),
    'pred_bond_length': res_reactot[1].numpy(),
    'bond_type': res_reactot[2].numpy()})
res_ours = pd.DataFrame({
    'true_bond_length': res_ours[0].numpy(),
    'pred_bond_length': res_ours[1].numpy(),
    'bond_type': res_ours[2].numpy()})

with pd.ExcelWriter('res_bond_analysis.xlsx') as writer:
    res_neuralNEB.to_excel(writer, sheet_name='NeuralNEB', index=False)
    res_learnTS.to_excel(writer, sheet_name='PSI_Based', index=False)
    res_reactdiff.to_excel(writer, sheet_name='ReactDiff', index=False)
    res_reactot.to_excel(writer, sheet_name='ReactOT', index=False)
    res_ours.to_excel(writer, sheet_name='TS-DFM', index=False)